In [ ]:
import arcpy
import pandas as pd
import numpy as np
import re

In [ ]:
# setting workspace
arcpy.env.workspace = "YOUR/PATH/TO/industrial-pollution-risk.gdb"
arcpy.env.overwriteOutput = True

## Most at-risk schools
The schools with the highest underlying toxicity concentration values.

In [3]:
# converting to dataframe
SCH_rsei = pd.DataFrame(arcpy.da.FeatureClassToNumPyArray("SCH_rsei", [
    "LEAID",
    "NAME",
    "CITY",
    "STATE",
    "ZIP",
    "LAT",
    "LON",
    "ToxConc",
    "ToxConcP"]
))

# cleaning & sorting by underlying toxconc
SCH_rsei = SCH_rsei.rename(columns={"NAME": "Name"})
SCH_rsei.Name = SCH_rsei.Name.str.title()
SCH_rsei["Location"] = SCH_rsei.CITY.str.title() + ", " + SCH_rsei.STATE
SCH_rsei = SCH_rsei.sort_values("ToxConc", ascending=False)

# exporting top 10 schools
SCH_rsei[["Name", "Location", "ToxConc", "ToxConcP"]][:10].to_csv("top10sch.csv")

# printing top 30 schools
SCH_rsei[:30]

,LEAID,Name,CITY,STATE,ZIP,LAT,LON,ToxConc,ToxConcP,Location
101632,7200030,Antonio Gonzalez Suarez,ANASCO,PR,00610,18.287501,-67.139382,6961550.0,0.999988,"Anasco, PR"
89619,4835430,Groves Middle,GROVES,TX,77619,29.957261,-93.925086,2538810.0,0.999976,"Groves, TX"
89620,4835430,Port Neches Middle,PORT NECHES,TX,77651,29.979621,-93.956789,2133800.0,0.999965,"Port Neches, TX"
89621,4835430,Port Neches-Groves H S,PORT NECHES,TX,77651,29.988760,-93.953353,2133800.0,0.999965,"Port Neches, TX"
89626,4835430,Port Neches Int,PORT NECHES,TX,77651,29.997614,-93.962339,2133800.0,0.999965,"Port Neches, TX"
89622,4835430,Alter Sch,PORT NECHES,TX,77651,29.990954,-93.966162,2133800.0,0.999965,"Port Neches, TX"
87739,4826190,The Academy Of Viola Dewalt H S,LA PORTE,TX,77571,29.670033,-95.022115,1998590.0,0.999953,"La Porte, TX"
3782,0405030,Lee Kornegay Intermediate School,Miami,AZ,85539,33.410774,-110.833222,1930280.0,0.999941,"Miami, AZ"
3781,0405030,Miami Junior Senior High School,Miami,AZ,85539,33.410774,-110.833222,1930280.0,0.999941,"Miami, AZ"
3783,0405030,M.U.S.D. #40 - Little Vandal Preschool,MIAMI,AZ,85539,33.408053,-110.836538,1930280.0,0.999941,"Miami, AZ"


In [4]:
# top 10 cities where there are the most schools in the 95th percentile of pollution
SCH_rsei[SCH_rsei.ToxConcP>=0.95].groupby(["Location"]).count()["Name"].sort_values(ascending=False)[:10]

Location
Houston, TX        560
Chicago, IL        456
Portland, OR       146
Memphis, TN        105
Richmond, VA       105
Baton Rouge, LA    102
Kansas City, MO     89
Reno, NV            80
Pasadena, TX        49
Kansas City, KS     45
Name: Name, dtype: int64

In [5]:
# top 10 states where there are the most schools in the 95th percentile of pollution
SCH_rsei[SCH_rsei.ToxConcP>=0.95].groupby("STATE").count()["Name"].sort_values(ascending=False)[:10]

STATE
TX    1185
IL     612
LA     240
VA     232
OR     223
PA     218
IN     175
OH     174
TN     161
NV     137
Name: Name, dtype: int64

## Worst polluters within 5 km of a school

### Worst industrial facilities
The top facilities operating within 5 km of a school (ranked by emissions).

In [6]:
# converting layer of sites within 5km of a school to a dataframe
TRI_5km = pd.DataFrame(arcpy.da.FeatureClassToNumPyArray("TRI_5km", [
    "TRIFD",
    "FACILITY_NAME", 
    "CITY",
    "ST",
    "ON_SITE_RELEASE_TOTAL", 
    "STANDARD_PARENT_CO_NAME",
    "FEDERAL_FACILITY",
    "INDUSTRY_SECTOR"]
))

# sorting by total on-site releases
TRI_5km = TRI_5km.sort_values('ON_SITE_RELEASE_TOTAL', ascending=False)
TRI_5km['Location'] = TRI_5km.CITY.str.title() + ', ' + TRI_5km.ST

# cleaning list of top 10 sites
top10sites = TRI_5km[['FACILITY_NAME', 'Location','ON_SITE_RELEASE_TOTAL', 'INDUSTRY_SECTOR']][:10]
top10sites = top10sites.rename(columns={"FACILITY_NAME":"Facility", "ON_SITE_RELEASE_TOTAL":"Emissions", "INDUSTRY_SECTOR":"Sector"})
top10sites.Facility = top10sites.Facility.str.title()
top10sites['Emissions'] = (np.round(top10sites['Emissions']/2000/1e3)).astype(int) # to tons
top10sites['Emissions'] = [str(x) + 'k tons' for x in top10sites['Emissions']]

# exporting and printing top 10 sites
top10sites.to_csv('top10sites.csv')
top10sites

,Facility,Location,Emissions,Sector
16733,Red Dog Operations,"Kotzebue, AK",329k tons,Metal Mining
15217,Nevada Gold Mines Llc - Goldstrike Mines Inc,"Carlin, NV",105k tons,Metal Mining
14653,Kennecott Utah Copper Mine Concentrators & Pow...,"Bingham Canyon, UT",66k tons,Metal Mining
16736,Hecla Greens Creek Mining Co,"Juneau, AK",45k tons,Metal Mining
15220,Nevada Gold Mines Llc - Carlin South Area,"Carlin, NV",21k tons,Metal Mining
15008,Freeport-Mcmoran Miami Inc,"Claypool, AZ",14k tons,Primary Metals
15218,Nevada Gold Mines Llc - Phoenix Mine,"Battle Mountain, NV",12k tons,Metal Mining
4693,Ascend Performance Materials Operations Llc,"Cantonment, FL",10k tons,Chemicals
14654,Kennecott Utah Copper Smelter & Refinery,"Magna, UT",10k tons,Primary Metals
13745,Ascend Performance Materials-Chocolate Bayou P...,"Alvin, TX",9k tons,Chemicals


### Worst parent companies
The top parent companies of facilities operating within 5 km of a school (ranked by emissions).

In [7]:
# converting total emissions across all sites to a dataframe
TRI_XY = pd.DataFrame(arcpy.da.FeatureClassToNumPyArray("TRI_XY", [
    "TRIFD",
    "FACILITY_NAME", 
    "ON_SITE_RELEASE_TOTAL", 
    "STANDARD_PARENT_CO_NAME",
    "FEDERAL_FACILITY",
    "INDUSTRY_SECTOR"]
))

# grouping by parent company and sorting by total on-site releases
comps = TRI_XY.groupby('STANDARD_PARENT_CO_NAME')['ON_SITE_RELEASE_TOTAL'].agg(
    Emissions='sum', Facilities='count').sort_values('Emissions', ascending=False)

In [8]:
# grouping <5 km site data set by parent company and sorting by total on-site releases
comps_5km = TRI_5km.groupby('STANDARD_PARENT_CO_NAME')['ON_SITE_RELEASE_TOTAL'].agg(
    Emissions='sum', Facilities='count').sort_values('Emissions', ascending=False)

# joining top polluter data with full country data to calculate share of emissions and facilities in the <5 km zone
comps_5km = comps_5km.join(comps, rsuffix=' total')
comps_5km['% of Total Emissions'] = comps_5km['Emissions'] / comps_5km['Emissions total'] * 100
comps_5km['% of Total Facilities'] = comps_5km['Facilities'] / comps_5km['Facilities total'] * 100
comps_5km = comps_5km[['Emissions', 'Facilities', '% of Total Emissions', '% of Total Facilities']]

# cleaning list of top 10 companies
top10comps = comps_5km[:10]
top10comps.index = [re.sub(' Inc$| Co$| Corp$| Llc$| Holdings$', '', x.title()) for x in top10comps.index]
top10comps.loc[:,'Emissions'] = (np.round(top10comps['Emissions']/2000/1e3)).astype(int) # to tons
top10comps.loc[:,'Emissions'] = [str(x) + 'k tons' for x in top10comps['Emissions']]
top10comps.loc[:,'% of Total Emissions'] = np.round(top10comps['% of Total Emissions']).astype(int)
top10comps.loc[:,'% of Total Emissions'] = [str(x) + '%' for x in top10comps['% of Total Emissions']]
top10comps.loc[:,'% of Total Facilities'] = np.round(top10comps['% of Total Facilities']).astype(int)
top10comps.loc[:,'% of Total Facilities'] = [str(x) + '%' for x in top10comps['% of Total Facilities']]

# exporting and printing top 10 companies
top10comps.to_csv('top10comps.csv')
top10comps

,Emissions,Facilities,% of Total Emissions,% of Total Facilities
Teck American,329.0k tons,2,100.0%,100.0%
Barrick Nevada Holding,138.0k tons,5,89.0%,56.0%
Rio Tinto America,76.0k tons,3,100.0%,100.0%
Hecla Mining,53.0k tons,2,100.0%,100.0%
Freeport-Mcmoran,21.0k tons,7,85.0%,58.0%
Ascend Performance Materials Holdings,20.0k tons,5,100.0%,100.0%
International Paper,13.0k tons,18,70.0%,72.0%
Nyrstar Us,13.0k tons,6,100.0%,100.0%
The Chemours,12.0k tons,11,98.0%,69.0%
Clean Harbors,11.0k tons,108,53.0%,94.0%
